# M1-03 — CNN 3 blocks

Code tương ứng với `m1_03.py`. Chạy các cell từ trên xuống: import, cấu hình, xử lý ảnh, load dữ liệu, tạo model, train/validation và test.

Train tối đa 50 epoch, dừng khi validation loss không giảm trong 8 epoch liên tiếp. Test dùng trọng số có validation loss thấp nhất giữ trong RAM. Notebook chưa được chạy.


In [1]:
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report

from metrics import calculate_metrics

In [2]:
# 1. Cấu hình
DATA_DIR = "../../data/VN_trash_classification_preprocessing"
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
# 2. Xử lý ảnh
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])



In [4]:
# 3. Load dữ liệu
train_dataset = ImageFolder(DATA_DIR + "/train", transform=transform)
val_dataset = ImageFolder(DATA_DIR + "/val", transform=transform)
test_dataset = ImageFolder(DATA_DIR + "/test", transform=transform)

# Nạp ảnh bằng nhiều tiến trình để GPU không phải chờ CPU
loader_options = {"num_workers": 8, "pin_memory": True, "persistent_workers": True}
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, **loader_options)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))




Train: 9542
Validation: 1685
Test: 864


In [5]:
# 4. Tạo CNN 3 blocks
class CNN(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        self.layers = nn.Sequential(
            # [B, 3, 224, 224] -> [B, 32, 112, 112]
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # [B, 32, 112, 112] -> [B, 64, 56, 56]
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # [B, 64, 56, 56] -> [B, 128, 28, 28]
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # [B, 128, 28, 28] -> [B, 128, 4, 4] -> [B, 2048]
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, number_of_classes)
        )

    def forward(self, images):
        return self.layers(images)


model = CNN(len(train_dataset.classes)).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)




In [6]:
# 5. Train và Validation
best_val_loss = float("inf")
best_weights = None
best_epoch = 0
patience = 15
epochs_without_improvement = 0

for epoch in range(50):
    model.train()
    train_loss = 0
    train_true_labels = []
    train_predicted_labels = []

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_function(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        predictions = outputs.argmax(dim=1)

        train_true_labels.extend(labels.cpu().tolist())
        train_predicted_labels.extend(predictions.cpu().tolist())

    train_loss /= len(train_dataset)
    train_metrics = calculate_metrics(train_true_labels, train_predicted_labels)
    train_accuracy = train_metrics["accuracy"]

    model.eval()
    val_loss = 0
    val_true_labels = []
    val_predicted_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_function(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)

            val_true_labels.extend(labels.cpu().tolist())
            val_predicted_labels.extend(predictions.cpu().tolist())

    val_loss /= len(val_dataset)
    val_metrics = calculate_metrics(val_true_labels, val_predicted_labels)
    val_accuracy = val_metrics["accuracy"]

    # Sao chép trọng số sang RAM để các epoch sau không ghi đè lên bản tốt nhất.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        best_epoch = epoch + 1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1}/50 | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2%} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2%}"
    )

    # Dừng khi validation loss không giảm trong 15 epoch liên tiếp.
    if epochs_without_improvement >= patience:
        print(f"Early stopping tại epoch {epoch + 1}")
        break

# Vòng lặp kết thúc khi model đang giữ trọng số của epoch cuối, nạp lại trọng số tốt nhất.
model.load_state_dict(best_weights)




Epoch 1/50 | Train Loss: 1.8401 | Train Acc: 34.27% | Val Loss: 2.0150 | Val Acc: 24.69%


Epoch 2/50 | Train Loss: 1.6280 | Train Acc: 42.88% | Val Loss: 1.5745 | Val Acc: 44.27%


Epoch 3/50 | Train Loss: 1.5338 | Train Acc: 46.22% | Val Loss: 2.0011 | Val Acc: 34.78%


Epoch 4/50 | Train Loss: 1.4572 | Train Acc: 49.16% | Val Loss: 1.6179 | Val Acc: 39.82%


Epoch 5/50 | Train Loss: 1.3946 | Train Acc: 51.77% | Val Loss: 2.4342 | Val Acc: 27.54%


...

Epoch 28/50 | Train Loss: 0.8005 | Train Acc: 72.21% | Val Loss: 1.3268 | Val Acc: 57.21%


Epoch 29/50 | Train Loss: 0.7857 | Train Acc: 73.01% | Val Loss: 1.5466 | Val Acc: 52.40%


Epoch 30/50 | Train Loss: 0.7781 | Train Acc: 72.88% | Val Loss: 1.2817 | Val Acc: 61.48%


Epoch 31/50 | Train Loss: 0.7460 | Train Acc: 73.70% | Val Loss: 1.2028 | Val Acc: 62.14%
Early stopping tại epoch 31


<All keys matched successfully>

In [7]:
# 6. Test
print(f"Best epoch: {best_epoch} | Best Val Loss: {best_val_loss:.6f}")
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

metrics = calculate_metrics(true_labels, predicted_labels)

print("\nTest Result")
print(f"Test Accuracy: {metrics['accuracy']:.2%}")
print(f"Macro Precision: {metrics['macro_precision']:.4f}")
print(f"Macro Recall: {metrics['macro_recall']:.4f}")
print(f"Macro F1: {metrics['macro_f1']:.4f}")
print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
print()
print(classification_report(true_labels, predicted_labels, target_names=test_dataset.classes, digits=4, zero_division=0))

Best epoch: 16 | Best Val Loss: 1.197410



Test Result
Test Accuracy: 48.73%
Macro Precision: 0.5196
Macro Recall: 0.4879
Macro F1: 0.4840
Weighted F1: 0.4920

              precision    recall  f1-score   support

         Alu     0.5575    0.6364    0.5943        99
      Carton     0.8333    0.6019    0.6989       108
    Foam_box     0.6087    0.3218    0.4211        87
    Milk_box     0.6614    0.8317    0.7368       101
       Other     0.2826    0.5200    0.3662        50
         PET     0.2695    0.3725    0.3128       102
       Paper     0.6064    0.5377    0.5700       106
   Paper_cup     0.6275    0.3048    0.4103       105
 Plastic_cup     0.2295    0.2642    0.2456       106

    accuracy                         0.4873       864
   macro avg     0.5196    0.4879    0.4840       864
weighted avg     0.5336    0.4873    0.4920       864



In [8]:
# 7. Lưu model
torch.save(model.state_dict(), CHECKPOINT_DIR + "/m1_03.pt")